# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an MLCroissantMetadata object

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema structure.

We will list all available record sets, their `@id`s, field `@id`s, and a sample of the columns present. All items are identified by their `@id` as per the Croissant schema.

In [ ]:
# List available record sets and fields
print('== Available Record Sets ==')
record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in getattr(metadata, 'recordSet', [])]
if not record_set_ids:
    # Try fallback for known id
    record_set_ids = ['https://api.app.sen.science/frontiers/7154140/1cfd3c55-7469-4d6f-97ad-4d7c61c7da05'] # Example fallback
    print('Warning: No record sets found in metadata, using fallback test id.')

for rid in record_set_ids:
    print(f"- RecordSet @id: {rid}")
    try:
        rs = dataset.metadata.entity_from_id(rid)
        if hasattr(rs, 'field'):
            fields = rs.field
            field_ids = [f['@id'] if isinstance(f, dict) else f for f in fields]
            print(f"  Fields: {field_ids}")
        else:
            print('  (No fields attribute in this record set)')
    except Exception as e:
        print(f'  (Error loading record set details: {e})')

In the following cell, we print a few sample records from the main record set. Make sure to reference the record set by its `@id`.

In [ ]:
# Print a few sample records for a selected record set
selected_record_set = record_set_ids[0]  # Use the first detected record set
print(f'Sample records for record set: {selected_record_set}')
try:
    for i, record in enumerate(dataset.records(record_set=selected_record_set)):
        print(record)
        if i >= 2:
            break
except Exception as e:
    print(f'Error iterating sample records: {e}')

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Extract data from each record set
dataframes = {}
print(f'Loading data for record sets: {record_set_ids}')

for record_set in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f'Columns in record set {record_set}:')
        print(df.columns.tolist())
    except Exception as e:
        print(f'Could not load records for {record_set}: {e}')

# View top rows of the primary record set
main_record_set = record_set_ids[0]
if main_record_set in dataframes:
    display_cols = dataframes[main_record_set].columns.tolist()
    print('Columns:', display_cols)
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

_Note: The actual field `@id`s and names depend on the schema. Below we provide an example using column names as loaded above. You may need to adapt these for the actual schema or preview the loaded DataFrames._

In [ ]:
# Set up EDA for the main record set
import numpy as np

# Choose a numeric field for demonstration (try to auto-detect or fall back to a common one)
main_df = dataframes.get(main_record_set)
if main_df is not None:
    col_sample = main_df.dtypes[main_df.dtypes.apply(lambda x: np.issubdtype(x, np.number))].index.tolist()
    if col_sample:
        numeric_field_id = col_sample[0]  # use first numeric
        print(f'Using numeric field for analysis: {numeric_field_id}')
    else:
        numeric_field_id = main_df.columns[0]
        print(f'No clear numeric field, using first column: {numeric_field_id}')

    # Filtering: e.g., keep records where numeric_field_id > threshold
    threshold = main_df[numeric_field_id].mean() if np.issubdtype(main_df[numeric_field_id].dtype, np.number) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold] if np.issubdtype(main_df[numeric_field_id].dtype, np.number) else main_df
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization example
    norm_col = f"{numeric_field_id}_normalized"
    if np.issubdtype(main_df[numeric_field_id].dtype, np.number):
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a text/categorical field
    group_field = None
    string_cols = main_df.dtypes[main_df.dtypes == object].index.tolist()
    if string_cols:
        group_field = string_cols[0]
        print(f'Attempting to group by: {group_field}')
    if group_field and group_field in main_df.columns and np.issubdtype(main_df[numeric_field_id].dtype, np.number):
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Example visualization for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and np.issubdtype(main_df[numeric_field_id].dtype, np.number):
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Scatterplot with group field if relevant
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.show()

## 6. Conclusion
The Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset provides a rich resource for the exploration of protein abundance and properties. Using `mlcroissant`, we've loaded metadata, reviewed record sets and fields by their unique `@id`, loaded records into DataFrames, performed basic exploratory data analysis, and visualized key numeric fields.

Further analysis (such as deep statistical modeling, feature engineering, or machine/deep learning tasks) can follow depending on the scientific or clinical research questions for your workflow.